In [3]:
## IMPORTS & CONNECTION
import pandas as pd
import os
import sys
from sqlalchemy import create_engine, text

sys.path.insert(0, os.path.abspath('..'))
import config

# Build the connection string and test it
engine = create_engine(
    f"mysql+pymysql://{config.DB_CONFIG['user']}:{config.DB_CONFIG['password']}"
    f"@{config.DB_CONFIG['host']}:{config.DB_CONFIG['port']}/{config.DB_CONFIG['database']}",
    echo=False
)

with engine.connect() as conn:
    result = conn.execute(text("SELECT DATABASE()"))
    print(f"Connected to: {result.fetchone()[0]}")

Connected to: stockvision


In [4]:
## CREATE TABLES
create_stocks = """
CREATE TABLE IF NOT EXISTS stocks (
    stock_id     INT                  AUTO_INCREMENT PRIMARY KEY,
    ticker       VARCHAR(20)          NOT NULL UNIQUE,
    company_name VARCHAR(100)         NOT NULL,
    market       ENUM('India', 'US')  NOT NULL,
    sector       VARCHAR(50)          NOT NULL,
    exchange     VARCHAR(10)          NOT NULL,
    created_at   TIMESTAMP            DEFAULT CURRENT_TIMESTAMP
)
"""

create_prices = """
CREATE TABLE IF NOT EXISTS daily_prices (
    price_id     BIGINT               AUTO_INCREMENT PRIMARY KEY,
    stock_id     INT                  NOT NULL,
    ticker       VARCHAR(20)          NOT NULL,
    market       ENUM('India', 'US')  NOT NULL,
    trade_date   DATE                 NOT NULL,
    open_price   DECIMAL(12,4),
    high_price   DECIMAL(12,4),
    low_price    DECIMAL(12,4),
    close_price  DECIMAL(12,4),
    volume       BIGINT,
    FOREIGN KEY  (stock_id) REFERENCES stocks(stock_id) ON DELETE CASCADE,
    UNIQUE KEY   uq_ticker_date  (ticker, trade_date),
    INDEX        idx_ticker      (ticker),
    INDEX        idx_market_date (market, trade_date),
    INDEX        idx_trade_date  (trade_date)
)
"""

with engine.connect() as conn:
    conn.execute(text(create_stocks))
    conn.execute(text(create_prices))
    # TRUNCATE makes re-running this notebook safe
    conn.execute(text("SET FOREIGN_KEY_CHECKS = 0"))
    conn.execute(text("TRUNCATE TABLE daily_prices"))
    conn.execute(text("TRUNCATE TABLE stocks"))
    conn.execute(text("SET FOREIGN_KEY_CHECKS = 1"))
    conn.commit()

print("Tables ready — stocks and daily_prices created and cleared.")

Tables ready — stocks and daily_prices created and cleared.


In [5]:
## STOCKS MASTER TABLE
COMPANY_NAMES = {
    # ── NIFTY 50 ─────────────────────────────────────────────
    "TCS.NS":"Tata Consultancy Services", "INFY.NS":"Infosys",
    "HCLTECH.NS":"HCL Technologies",      "WIPRO.NS":"Wipro",
    "TECHM.NS":"Tech Mahindra",           "HDFCBANK.NS":"HDFC Bank",
    "ICICIBANK.NS":"ICICI Bank",          "AXISBANK.NS":"Axis Bank",
    "KOTAKBANK.NS":"Kotak Mahindra Bank", "SBIN.NS":"State Bank of India",
    "INDUSINDBK.NS":"IndusInd Bank",      "BAJFINANCE.NS":"Bajaj Finance",
    "BAJAJFINSV.NS":"Bajaj Finserv",      "HDFCLIFE.NS":"HDFC Life Insurance",
    "SBILIFE.NS":"SBI Life Insurance",    "SHRIRAMFIN.NS":"Shriram Finance",
    "MARUTI.NS":"Maruti Suzuki",          "TATAMOTORS.NS":"Tata Motors",
    "M&M.NS":"Mahindra & Mahindra",       "BAJAJ-AUTO.NS":"Bajaj Auto",
    "HEROMOTOCO.NS":"Hero MotoCorp",      "EICHERMOT.NS":"Eicher Motors",
    "SUNPHARMA.NS":"Sun Pharmaceutical",  "DRREDDY.NS":"Dr. Reddy's Laboratories",
    "CIPLA.NS":"Cipla",                   "DIVISLAB.NS":"Divi's Laboratories",
    "APOLLOHOSP.NS":"Apollo Hospitals",   "RELIANCE.NS":"Reliance Industries",
    "ONGC.NS":"ONGC",                     "BPCL.NS":"Bharat Petroleum Corporation",
    "POWERGRID.NS":"Power Grid Corporation","NTPC.NS":"NTPC",
    "COALINDIA.NS":"Coal India",          "HINDUNILVR.NS":"Hindustan Unilever",
    "ITC.NS":"ITC",                       "NESTLEIND.NS":"Nestle India",
    "BRITANNIA.NS":"Britannia Industries","TATACONSUM.NS":"Tata Consumer Products",
    "TATASTEEL.NS":"Tata Steel",          "HINDALCO.NS":"Hindalco Industries",
    "JSWSTEEL.NS":"JSW Steel",            "LT.NS":"Larsen & Toubro",
    "ADANIPORTS.NS":"Adani Ports & SEZ",  "ADANIENT.NS":"Adani Enterprises",
    "ASIANPAINT.NS":"Asian Paints",       "GRASIM.NS":"Grasim Industries",
    "ULTRACEMCO.NS":"UltraTech Cement",   "TITAN.NS":"Titan Company",
    "BHARTIARTL.NS":"Bharti Airtel",      "BEL.NS":"Bharat Electronics",
    # ── S&P 500 Top 30 ───────────────────────────────────────
    "AAPL":"Apple",               "NVDA":"NVIDIA",
    "MSFT":"Microsoft",           "AVGO":"Broadcom",
    "AMD":"Advanced Micro Devices","ORCL":"Oracle",
    "CSCO":"Cisco Systems",       "CRM":"Salesforce",
    "AMZN":"Amazon",              "TSLA":"Tesla",
    "HD":"Home Depot",            "MCD":"McDonald's",
    "COST":"Costco",              "GOOGL":"Alphabet (Class A)",
    "META":"Meta Platforms",      "NFLX":"Netflix",
    "JPM":"JPMorgan Chase",       "V":"Visa",
    "MA":"Mastercard",            "BAC":"Bank of America",
    "BRK-B":"Berkshire Hathaway (Class B)", "LLY":"Eli Lilly",
    "UNH":"UnitedHealth Group",   "JNJ":"Johnson & Johnson",
    "ABT":"Abbott Laboratories",  "XOM":"Exxon Mobil",
    "CVX":"Chevron",              "WMT":"Walmart",
    "PG":"Procter & Gamble",      "CAT":"Caterpillar",
}

EXCHANGES = {t: "NSE" for t in config.NIFTY_50}
EXCHANGES.update({
    "AAPL":"NASDAQ","NVDA":"NASDAQ","MSFT":"NASDAQ","AVGO":"NASDAQ",
    "AMD":"NASDAQ", "ORCL":"NYSE", "CSCO":"NASDAQ","CRM":"NYSE",
    "AMZN":"NASDAQ","TSLA":"NASDAQ","HD":"NYSE",   "MCD":"NYSE",
    "COST":"NASDAQ","GOOGL":"NASDAQ","META":"NASDAQ","NFLX":"NASDAQ",
    "JPM":"NYSE",  "V":"NYSE",    "MA":"NYSE",   "BAC":"NYSE",
    "BRK-B":"NYSE","LLY":"NYSE",  "UNH":"NYSE",  "JNJ":"NYSE",
    "ABT":"NYSE",  "XOM":"NYSE",  "CVX":"NYSE",  "WMT":"NYSE",
    "PG":"NYSE",   "CAT":"NYSE",
})

# Reverse sector lookup from config
def get_sector(ticker, market):
    sectors = config.NIFTY_SECTORS if market == 'India' else config.SP500_SECTORS
    for sector, tickers in sectors.items():
        if ticker in tickers:
            return sector
    return 'Other'

rows = []
for market_key, tickers in config.ALL_STOCKS.items():
    market_label = 'India' if market_key == 'india' else 'US'
    for ticker in tickers:
        rows.append({
            'ticker':       ticker,
            'company_name': COMPANY_NAMES.get(ticker, ticker),
            'market':       market_label,
            'sector':       get_sector(ticker, market_label),
            'exchange':     EXCHANGES.get(ticker, 'Unknown')
        })

stocks_df = pd.DataFrame(rows)
stocks_df.to_sql('stocks', engine, if_exists='append', index=False)
print(f"Inserted {len(stocks_df)} stocks into stocks table")
print(stocks_df[['ticker','company_name','market','sector']].head(5).to_string())

Inserted 80 stocks into stocks table
       ticker               company_name market sector
0      TCS.NS  Tata Consultancy Services  India     IT
1     INFY.NS                    Infosys  India     IT
2  HCLTECH.NS           HCL Technologies  India     IT
3    WIPRO.NS                      Wipro  India     IT
4    TECHM.NS              Tech Mahindra  India     IT


In [6]:
## LOAD DAILY PRICES
def safe_filename(ticker):
    return ticker.replace("&", "AND").replace(".", "_").replace("-", "_")

# Fetch stock_id mapping from stocks table
with engine.connect() as conn:
    result  = conn.execute(text("SELECT stock_id, ticker FROM stocks"))
    id_map  = {row.ticker: row.stock_id for row in result}

total_rows = 0
errors     = []

for market_key, tickers in config.ALL_STOCKS.items():
    market_label = 'India' if market_key == 'india' else 'US'
    raw_dir      = f"../data/raw/{market_key}"

    for ticker in tickers:
        filepath = os.path.join(raw_dir, f"{safe_filename(ticker)}.csv")

        if not os.path.exists(filepath):
            print(f"  [SKIP]  {ticker:<22} — CSV not found")
            errors.append(ticker)
            continue

        try:
            df = pd.read_csv(filepath, index_col=0, parse_dates=True)
            df.index.name = 'trade_date'

            # Add metadata columns
            df['stock_id'] = id_map[ticker]
            df['ticker']   = ticker
            df['market']   = market_label

            # Standardise column names
            df = df.rename(columns={
                'Open':   'open_price',
                'High':   'high_price',
                'Low':    'low_price',
                'Close':  'close_price',
                'Volume': 'volume'
            })

            # Keep only the columns MySQL expects
            df = df[['stock_id','ticker','market',
                      'open_price','high_price','low_price','close_price','volume']]

            # Insert — chunksize=500 keeps memory usage low
            df.to_sql('daily_prices', engine, if_exists='append',
                      index=True, index_label='trade_date',
                      chunksize=500, method='multi')

            total_rows += len(df)
            print(f"  [OK]    {ticker:<22} | {len(df):>4} rows")

        except Exception as e:
            print(f"  [FAIL]  {ticker:<22} | {e}")
            errors.append(ticker)

print(f"\n  Total rows inserted : {total_rows:,}")
if errors:
    print(f"  Errors              : {errors}")

  [OK]    TCS.NS                 | 3699 rows
  [OK]    INFY.NS                | 3699 rows
  [OK]    HCLTECH.NS             | 3700 rows
  [OK]    WIPRO.NS               | 3699 rows
  [OK]    TECHM.NS               | 3699 rows
  [OK]    HDFCBANK.NS            | 3699 rows
  [OK]    ICICIBANK.NS           | 3699 rows
  [OK]    AXISBANK.NS            | 3699 rows
  [OK]    KOTAKBANK.NS           | 3699 rows
  [OK]    SBIN.NS                | 3699 rows
  [OK]    INDUSINDBK.NS          | 3699 rows
  [OK]    BAJFINANCE.NS          | 3699 rows
  [OK]    BAJAJFINSV.NS          | 3699 rows
  [OK]    HDFCLIFE.NS            | 1756 rows
  [OK]    SBILIFE.NS             | 1788 rows
  [OK]    SHRIRAMFIN.NS          | 3699 rows
  [OK]    MARUTI.NS              | 3699 rows
  [SKIP]  TATAMOTORS.NS          — CSV not found
  [OK]    M&M.NS                 | 3699 rows
  [OK]    BAJAJ-AUTO.NS          | 3699 rows
  [OK]    HEROMOTOCO.NS          | 3699 rows
  [OK]    EICHERMOT.NS           | 3699 rows
  [OK]

In [7]:
## VERIFICATION
with engine.connect() as conn:

    total = conn.execute(text("SELECT COUNT(*) FROM daily_prices")).fetchone()[0]
    print(f"Total rows in daily_prices : {total:,}")

    print("\nRows by market:")
    rows = conn.execute(text("""
        SELECT market,
               COUNT(*)             AS total_rows,
               COUNT(DISTINCT ticker) AS stocks
        FROM daily_prices
        GROUP BY market
    """))
    for r in rows:
        print(f"  {r.market:<6} : {r.total_rows:>8,} rows across {r.stocks} stocks")

    r = conn.execute(text("SELECT MIN(trade_date), MAX(trade_date) FROM daily_prices")).fetchone()
    print(f"\nDate range : {r[0]} → {r[1]}")

    print("\nSample — TCS.NS (first 3 rows):")
    rows = conn.execute(text("""
        SELECT trade_date, open_price, high_price, low_price, close_price, volume
        FROM daily_prices WHERE ticker = 'TCS.NS'
        ORDER BY trade_date LIMIT 3
    """))
    for r in rows:
        print(f"  {r.trade_date} | O:{r.open_price} H:{r.high_price} L:{r.low_price} C:{r.close_price} | Vol:{r.volume:,}")

    print("\nSample — AAPL (first 3 rows):")
    rows = conn.execute(text("""
        SELECT trade_date, open_price, close_price, volume
        FROM daily_prices WHERE ticker = 'AAPL'
        ORDER BY trade_date LIMIT 3
    """))
    for r in rows:
        print(f"  {r.trade_date} | O:{r.open_price} C:{r.close_price} | Vol:{r.volume:,}")

Total rows in daily_prices : 289,656

Rows by market:
  India  :  177,187 rows across 49 stocks
  US     :  112,469 rows across 30 stocks

Date range : 2010-01-04 → 2024-12-30

Sample — TCS.NS (first 3 rows):
  2010-01-04 | O:260.3456 H:261.7598 L:258.3624 C:259.2592 | Vol:1,963,682
  2010-01-05 | O:260.4145 H:261.9839 L:257.5000 C:259.3280 | Vol:2,014,488
  2010-01-06 | O:259.3281 H:259.4488 L:252.8263 C:253.4644 | Vol:3,349,176

Sample — AAPL (first 3 rows):
  2010-01-04 | O:6.3891 C:6.4065 | Vol:493,729,600
  2010-01-05 | O:6.4241 C:6.4176 | Vol:601,904,800
  2010-01-06 | O:6.4176 C:6.3155 | Vol:552,160,000
